# Persian TeCo — compact word EEG analysis
**CPU → Runtime → Run all**, then allow Drive mounting. Uses existing native reading signals, 126 channels at documented1200Hz. Code stays on GitHub; Persian text/labels and results stay private on Drive.

Requires private labels at `Thesis/Data/word_sentiment_eeg/teco/gpt uploaded files/word_labels.csv` and all12 existing native reader pickles. No dataset/model download. Original EEG is read into Colab memory one reader at a time. This is the first full TeCo extraction for this pipeline and may take substantially longer than the ZuCo saved-results review.

Saves to one folder: `Thesis/Results/word_sentiment_eeg/teco_v1_word_erp_compact`. Five group means: negative, pooled neutral, positive, neutral words in agent-reviewed strictly neutral sentences, and neutral words in original source-rated neutral sentences (source label0, which can still contain affective content). Controls and ambiguous/context-reversal words are excluded. It saves onsets, all126-channel averages, source checksums, counts, separate comparison rosters and scalp-region/subject plots. No individual-trial EEG archive. Valid checkpoints are reused without duplicate counting. At least300MiB mounted Drive free-space safety margin is required; actual output footprint depends on onset table size.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, sys, shutil

drive.mount('/content/drive')
THESIS = Path('/content/drive/MyDrive/Thesis')
REPO = Path('/content/word-sentiment-eeg-teco')
RUN_TAG = 'teco_v1_word_erp_compact'
LABELS = THESIS / 'Data/word_sentiment_eeg/teco/gpt uploaded files/word_labels.csv'
if not LABELS.is_file():
    raise FileNotFoundError(f'Private Persian word labels are not present: {LABELS}. Contact the project agent before extraction.')
def git(*args):
    subprocess.run(['git', '-c', 'credential.helper=', '-c', 'http.extraheader=', *map(str,args)], check=True, env=dict(os.environ,GIT_TERMINAL_PROMPT='0'))
if not REPO.exists():
    git('clone', '--depth', '1', 'https://github.com/parmisbathayan/word-sentiment-eeg.git', REPO)
else:
    if subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
        raise RuntimeError('Temporary TeCo checkout has edits. Preserve those edits before updating.')
    git('-C', REPO, 'pull', '--ff-only', 'origin', 'main')
subprocess.run([sys.executable,'-m','pip','install','-q','numpy','pandas','scipy','matplotlib'],check=True)
print('Code version:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','--short','HEAD'],text=True).strip())
print('Mounted Drive free GiB:',round(shutil.disk_usage(THESIS).free / 1024**3,2))
sys.path.insert(0,str(REPO/'scripts'))
from colab_progress import run_with_progress
command=[sys.executable,'-u',str(REPO/'run_teco_erp.py'),'--thesis-root',str(THESIS),'--run-tag',RUN_TAG]
run_with_progress(command,THESIS/'Results/word_sentiment_eeg'/RUN_TAG/'runtime_status.json',activity='Native TeCo word FRP extraction')


In [ ]:
from IPython.display import display, Markdown, Image
import json
import pandas as pd
RESULTS = THESIS / 'Results/word_sentiment_eeg' / RUN_TAG
print('Saved results:', RESULTS)
display(Markdown((RESULTS/'REPORT.md').read_text()))
display(pd.read_csv(RESULTS/'class_counts.csv'))
for name in ('pooled_neutral_regions.png','strict_neutral_regions.png','source_neutral_regions.png','sensor_regions.png'):
    path=RESULTS/'plots'/name
    if path.is_file():
        display(Image(filename=str(path)))
print('Subject plots are in',RESULTS/'plots')
